# Smart-G demo notebook for Objects

This is an interactive document allowing to run Smart-G with python and visualize the results. <br>
*Tips*: cells can be executed with shift-enter. Tooltips can be obtained with shift-tab. More information [here](http://ipython.org/notebook.html) or in the help menu. [A table of content can also be added](https://github.com/minrk/ipython_extensions#table-of-contents).

In [ ]:
%matplotlib inline
# next 2 lines allow to automatically reload modules that have been changed externally
%reload_ext autoreload
%autoreload 2

import sys
from pathlib import Path

try:
    import subprocess
    check = subprocess.check_call(['git', 'rev-parse', '--show-toplevel'],
                                  stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    ROOTPATH = subprocess.Popen(['git', 'rev-parse', '--show-toplevel'],
                                stdout=subprocess.PIPE).communicate()[0].rstrip().decode('utf-8') # Root Git Path
    ROOTPATH = Path(ROOTPATH)
except subprocess.CalledProcessError:
    ROOTPATH = Path.cwd()
sys.path.insert(0, str(ROOTPATH))

from smartg.config import DIR_AUXDATA
from smartg.smartg import Smartg
from smartg.objects3d import CusForward, CusBackward
from smartg.surface import LambSurface
from smartg.sensor import Sensor
from smartg.atmosphere import Atm1D, AerOPAC
from smartg.albedo import AlbedoCst
from smartg.objects3d import Mirror, Plane, Transformation, \
    Entity, Matte, generate_h_p, generate_h_a, extract_points, Heliostat
from smartg.view import smartg_view, receiver_view, cat_view, nopt_view, visualize_entity
from smartg.xarray import drop_axes
from luts.luts import Idx
import numpy as np

import warnings
warnings.simplefilter('always', DeprecationWarning)
import geoclide as gc

In [ ]:
# Uncomment below for 3D views
%matplotlib widget

# Quick Start

## Creation of objects (simple example)

In [ ]:
# We want to create a simple case with a receiver and four heliostats (MirA, B, C and D).
# The receiver will be created a x = 1km, the first heliostat at x = 950m, the
# second heliostat at x = 900m and so on. The mirror will be oriented such that the
# direct rays reflected by the mirrors will totally reach the receiver.

# The sun zenith angle
SZA = 14.3
# Create the heliostats MirA, MirB, MirC and MirD (smartg unit is the kilometer)
wMx = 0.004725
wMy = 0.00642
MirA = Entity(name = "reflector", \
              material_front = Mirror(reflectivity = 0.88), \
              material_back = Matte(), \
              geo = Plane( p1 = gc.Point(-wMx, -wMy, 0.),
                           p2 = gc.Point(wMx, -wMy, 0.),
                           p3 = gc.Point(-wMx, wMy, 0.),
                           p4 = gc.Point(wMx, wMy, 0.) ), \
              transformation = Transformation( rotation = np.array([0., 20.281725, 0.]), \
                                               translation = np.array([-0.05, 0., 0.00517]) ))

# MirB, C and D are exact copies of MirA, but with different transformations
MirB = Entity(MirA)
MirC = Entity(MirA)
MirD = Entity(MirA)
MirB.set_transformation(Transformation( rotation = np.array([0., 29.460753, 0.]), \
                                        translation = np.array([-0.1, 0., 0.00517]) ))
MirC.set_transformation(Transformation( rotation = np.array([0., 35.129831, 0.]), \
                                        translation = np.array([-0.15, 0., 0.00517]) ))
MirD.set_transformation(Transformation( rotation = np.array([0., 38.715473, 0.]), \
                                        translation = np.array([-0.2, 0., 0.00517]) ))

# Create the receiver Recept1
wRx = 0.006
wRy = 0.007
# tc = Taille cellule. The receiver is devided in several cells to visualize the flux distribution 
Rec1 = Entity(name = "receiver", tc = 0.0005, \
                 material_front = Matte(reflectivity = 0.), \
                 material_back = Matte(reflectivity = 0.), \
                 geo = Plane( p1 = gc.Point(-wRx, -wRy, 0.),
                              p2 = gc.Point(wRx, -wRy, 0.),
                              p3 = gc.Point(-wRx, wRy, 0.),
                              p4 = gc.Point(wRx, wRy, 0.) ), \
                 transformation = Transformation( rotation = np.array([0., -101.5, 0.]), \
                                                  translation = np.array([0., 0., 0.1065]) ))

# Create the list containing all the objects
LOBJ1 = [MirA, MirB, MirC, MirD, Rec1]

## How to verify the drawing of objects

In [ ]:
# 3D print of objects (the notebook give only a 2D print of the 3D one)
fig1 = visualize_entity(entities=[MirA, MirB, MirC, MirD, Rec1], theta_deg=SZA, 
                        show_rays=True, sr_view=1, rs_fac=1)

## Run the simulation

In [ ]:
# In this simulation the photons from TOA are launched to fill only the
# mirrors (ray tracing method or restricted forward method) --> cusForward = CusForward(lmode="RF")
# By default the sun is a ponctual sun source targeting the origin (0,0,0) --> --> cusForward = None
# For a full forward mode i.g. specify in Smartg.run() -> cusForward = CusForward(cfx=10, cfy=10, lmode="FF")
# Where CFX is the size in kilometer in the x axis of the rectangle. Be careful, the full forward mode 
# need a big number of photons to obtain a good accuracy !

w2 = 0.5
Pmin = [-w2, -w2, -0.005]
Pmax = [w2, w2, 0.125]
interval0 = [Pmin, Pmax] # enable to earn some computational time, can be very usefull in FF mode
aer = AerOPAC('desert', 0.25, 550.)
pro = Atm1D('afglms', comp=[aer], p0 = 877, tcwp=1.2)
m = Smartg(double = True, obj3d=True).run( surf = LambSurface(alb=AlbedoCst(0.25)), 
    th_v_deg=SZA, n_f=1e6, wavelength=550., nb_photons=4e7, nb_loop=5e6, atm=pro,
    my_objects=LOBJ1, interval = interval0, cus_l=CusForward(lmode="RF"))
m_ds = m


## How to show the results ?

In [ ]:
# Show the description of the simulation
print(m_ds)
print(m_ds.attrs)

In [ ]:
# print the infomation at the receiver
output = cat_view(m_ds)
print("\noutput:")
output

In [ ]:
# print the infomation at the receiver
output = cat_view(m_ds)
print("\noutput:")
output

In [ ]:
# print the total flux distribution at the receiver
receiver_view(ds_sg_out=m_ds)

In [ ]:
# print the flux distribution at the receiver from only the direct D (cat1)
receiver_view(ds_sg_out=m_ds, cat=1)

In [ ]:
# print the flux distribution at the receiver from beams from the process H only (cat2)
receiver_view(ds_sg_out=m_ds, cat=2)

In [ ]:
# print the flux distribution at the receiver from beams from the process E only (cat3)
receiver_view(ds_sg_out=m_ds, cat=3)

In [ ]:
# print the flux distribution at the receiver from beams from the process A only (cat4)
receiver_view(ds_sg_out=m_ds, cat=4)

In [ ]:
# print the flux distribution at the receiver from beams from the two processes H and A (cat5)
receiver_view(ds_sg_out=m_ds, cat=5)

In [ ]:
# print the flux distribution at the receiver from beams from the two processes H and E (cat6)
receiver_view(ds_sg_out=m_ds, cat=6)

In [ ]:
# print the flux distribution at the receiver from beams from the two processes E and A (cat7)
receiver_view(ds_sg_out=m_ds, cat=7)

In [ ]:
# print the flux distribution at the receiver from beams from the three processes H , E and A (cat8)
receiver_view(ds_sg_out=m_ds, cat=8)

# Complex cases

## Quick heliostat generation by giving two angles

In [ ]:
# Specify the sun zenith angle
SZA= 14.3

# Creation of a receiver
wRx = 0.006
wRy = 0.007

Rec2 = Entity(name = "receiver", tc = 0.0005, \
                 material_front = Matte(reflectivity = 0.), \
                 material_back = Matte(reflectivity = 0.), \
                 geo = Plane( p1 = gc.Point(-wRx, -wRy, 0.),
                              p2 = gc.Point(wRx, -wRy, 0.),
                              p3 = gc.Point(-wRx, wRy, 0.),
                              p4 = gc.Point(wRx, wRy, 0.) ), \
                 transformation = Transformation( rotation = np.array([0., -101.5, 0.]), \
                                                  translation = np.array([0., 0., 0.1065]) ))

pRe = gc.Point(Rec2.transformation.transx, Rec2.transformation.transy, Rec2.transformation.transz)

# Generation of heliostats thanks to two angles, min_ang_deg and max_ang_deg.
LOBJ2 = generate_h_a(theta_deg=SZA, receiver_pos=pRe, min_ang_deg=150, max_ang_deg=210, gap_ang_deg = 10,
                        first_dist=0.1, n_heliostats = 3, gap_dist = 0.008, helio_size_x = 0.005, helio_size_y = 0.01,
                        pillar_height= 0.00517, reflectivity=0.88)

# Without forgetting to add the reveicer at the list of objects
LOBJ2.append(Rec2)

### Drawing verification

In [ ]:
fig1 = visualize_entity(LOBJ2, theta_deg=SZA)

### Run the simulation

In [ ]:
aer = AerOPAC('desert', 1, 550.)
pro = Atm1D('afglms', comp=[aer])
lMode1 = CusForward(lmode="RF")

m2 = Smartg(double = True, obj3d=True).run( surf = LambSurface(alb=AlbedoCst(0.25)), 
        th_v_deg=SZA, n_f=1e6, wavelength=550., nb_photons=1e7, nb_loop=1e6, atm=pro,
        my_objects=LOBJ2, cus_l=lMode1)
m2_ds = m2


In [ ]:
print(m2_ds)
print(m2_ds.attrs)

### Results

In [ ]:
cat_view(m2_ds, accuracy=3)

In [ ]:
receiver_view(ds_sg_out=m2_ds, log_color_scale=True)

## Quick heliostat generation by giving positions from a file

In [ ]:
# You need a file with the x, y and z positions, see the file HPOS_STP1.dat as example
# Points of heliostats are extracted from the given file as a list of class point
lPH = extract_points(filename = DIR_AUXDATA / 'STPs' / 'STP1.dat')

# Specify the solar zenith angle
SZA = 14.3

# Creation of a receiver
wRx = 0.006
wRy = 0.007

Rec3 = Entity(name = "receiver", tc = 0.0005, \
                 material_front = Matte(reflectivity = 0.), \
                 material_back = Matte(reflectivity = 0.), \
                 geo = Plane( p1 = gc.Point(-wRx, -wRy, 0.),
                              p2 = gc.Point(wRx, -wRy, 0.),
                              p3 = gc.Point(-wRx, wRy, 0.),
                              p4 = gc.Point(wRx, wRy, 0.) ), \
                 transformation = Transformation( rotation = np.array([0., -101.5, 0.]), \
                                                  translation = np.array([0., 0., 0.1065]) ))

# Coordinate of the center of the created receiver needed for heliostat generation
pRe = gc.Point(Rec3.transformation.transx, Rec3.transformation.transy, Rec3.transformation.transz)

# Generate heliostats thanks to a list of Points, lPH.
LOBJ3 = generate_h_p(theta_deg = SZA, heliostat_pos_list = lPH, receiver_pos = pRe,
                        helio_size_x = 0.00945, helio_size_y = 0.01284, reflectivity = 0.88)

# Without forgetting to add the reveicer at the list of objects
LOBJ3.append(Rec3)

### Drawing verification

In [ ]:
fig1 = visualize_entity(LOBJ3, theta_deg=SZA)

### Run the simulation

In [ ]:
Pmin = [-0.12, -0.05, -0.05]
Pmax = [0.05, 0.05, 0.125]
interval0 = [Pmin, Pmax] # enable to earn some computational time, can be very usefull in FF mode
aer = AerOPAC('desert', 0.2, 550.)
pro = Atm1D('afglms', comp=[aer])
lMode = CusForward(cfx = 0.06, cfy = 0.08, cftx = -0.08, cfty = 0., lmode="FF")
m3 = Smartg(double = True, obj3d=True).run( surf = LambSurface(AlbedoCst(0.25)), 
        th_v_deg=SZA, n_f=1e6, wavelength=550., nb_photons=4e7, nb_loop=2e6, atm=pro,
        my_objects=LOBJ3, interval = interval0, cus_l=lMode)
m3_ds = m3


In [ ]:
print(m3_ds)
print(m3_ds.attrs)

### Results

In [ ]:
cat_view(m3_ds)

In [ ]:
receiver_view(ds_sg_out=m3_ds, vmin=0)

# More complex cases

## Deals with 3D object in backward mode

### Creation of the objects

In [ ]:
# Zenith and Azimuth angle of the sun, respectively Theta and phi
SZA = 50
SAA = 45
PHI = 180.-SAA

# Position of heliostats (list) [pH] and position of the sensor pS
pH = [gc.Point(-0.05, 0., 0.00517)] # here only one
pS = gc.Point(0., 0., 0.1065)

# create the heliostats : here we need the zenith and azimuth angles
LOBJ4 = generate_h_p(theta_deg = SZA, phi_deg = PHI, heliostat_pos_list = pH, receiver_pos = pS, 
                    helio_size_x = 0.00945, helio_size_y = 0.01284, reflectivity = 0.88)

# Modify the rugosity ? here -->
LOBJ4[0].material_front = Mirror(reflectivity = 0.88, roughness=0.1, shadow=True)

# Creation of the sensor
vS = gc.Vector(pH[0]-pS) # direction of the sensor described by a vector
SEN = Sensor(POSX = pS.x, POSY = pS.y, POSZ = pS.z, LOC  = 'ATMOS', FOV  = 0., TYPE = 0, V=vS)

### Run the simulation

In [ ]:
aer = AerOPAC('desert', 0.2, 550.)
pro = Atm1D('afglms', comp=[aer]).calc(550.)

m4 = Smartg(double = True, obj3d=True, back = True).run( surf = LambSurface(alb=AlbedoCst(0.25)),
                n_f=1e6, wavelength=550., nb_photons=1e8, nb_loop=1e7, atm=pro,
                my_objects=LOBJ4, sensor = SEN)

### See the results

In [ ]:
ind_az = [int(np.abs(m4['Azimuth angles'].values - a).argmin())
          for a in [0, 22, 44, 68, 90, 112, 134, 156]]
_=smartg_view(m4, qu=True, ind=ind_az)

## STP construction with heliostats composed of facets

In [ ]:
# Specify the sun zenith angle and sun azimuth angle
SZA = 50
SAA = 100
PHI = 180. - SAA

# Creation of a receiver
wRx = 0.006
wRy = 0.007

Rec5 = Entity(name = "receiver", tc = 0.0005, \
                 material_front = Matte(reflectivity = 0.), \
                 material_back = Matte(reflectivity = 0.), \
                 geo = Plane( p1 = gc.Point(-wRx, -wRy, 0.),
                              p2 = gc.Point(wRx, -wRy, 0.),
                              p3 = gc.Point(-wRx, wRy, 0.),
                              p4 = gc.Point(wRx, wRy, 0.) ), \
                 transformation = Transformation( rotation = np.array([0., -101.5, 0.]), \
                                                  translation = np.array([0., 0., 0.1065]) ))

pRe = gc.Point(Rec5.transformation.transx, Rec5.transformation.transy, Rec5.transformation.transz)

# Generation of heliostats with facets
H0 = Heliostat(n_facets_x = 2, n_facets_y = 2, helio_size_x = 0.00945, helio_size_y = 0.01284)
LOBJ5 = generate_h_a(theta_deg=SZA, phi_deg= PHI, receiver_pos=pRe, min_ang_deg=140, max_ang_deg=220, gap_ang_deg = 10, first_dist=0.15,
                        n_heliostats = 4, gap_dist = 0.03, pillar_height= 0.00517, reflectivity=0.88, roughness = 0.001, heliostat_type=H0)

# Without forgetting to add the reveicer at the list of objects
LOBJ5.append(Rec5)

### Drawing verification

In [ ]:
# visualize without facets
LOBJ5_bis = generate_h_a(theta_deg=SZA, phi_deg= PHI, receiver_pos=pRe, min_ang_deg=140, max_ang_deg=220, gap_ang_deg = 10, first_dist=0.15,
                        n_heliostats = 4, gap_dist = 0.03, pillar_height= 0.00517, helio_size_x = 0.00945, helio_size_y = 0.01284, reflectivity=0.88, roughness = 0.001)
LOBJ5_bis.append(Rec5)
fig1 = visualize_entity(LOBJ5_bis, theta_deg=SZA, phi_deg=PHI, show_rays=True)

### Run the simulation

In [ ]:
Pmin = [-0.6, -0.6, 0.]
Pmax = [0.6, 0.6, 0.125]
interval0 = [Pmin, Pmax] # enable to earn some computational time, can be very usefull in FF mode
aer = AerOPAC('desert', 0.5, 550.)
pro = Atm1D('afglms', comp=[aer])
lMode = CusForward(cfx = 8, cfy = 8, cftx = -0.1, cfty = 0., lmode="FF", sampling='isotropic', fov=0.266)
m5 = Smartg(double = True, obj3d=True).run( surf = LambSurface(alb=AlbedoCst(0.25)), 
        th_v_deg=SZA, ph_v_deg=PHI, n_f=1e6, wavelength=550., nb_photons=4e9, nb_loop=5e7, atm=pro,
        my_objects=LOBJ5, interval = interval0, cus_l=lMode, direct=True)
m5_ds = m5


### Results

In [ ]:
print(m5_ds)
print(m5_ds.attrs)

In [ ]:
print("kernel time(s)=", m5.attrs['kernel time (s)'])
cat_view(m5_ds, output_unit='FLUX', flux_unit='kW')

In [ ]:
receiver_view(ds_sg_out=m5_ds, vmin=0, flux_unit='kW')

## The Same in backward

In [ ]:
# Specify the sun zenith angle and sun azimuth angle
SZA = 50
SAA = 100
PHI = 180. - SAA

# Creation of a receiver
wRx = 0.006
wRy = 0.007

Rec6 = Entity(name = "receiver", tc = 0.0005, \
                 material_front = Matte(reflectivity = 0.), \
                 material_back = Matte(reflectivity = 0.), \
                 geo = Plane( p1 = gc.Point(-wRx, -wRy, 0.),
                              p2 = gc.Point(wRx, -wRy, 0.),
                              p3 = gc.Point(-wRx, wRy, 0.),
                              p4 = gc.Point(wRx, wRy, 0.) ), \
                 transformation = Transformation( rotation = np.array([0., -101.5, 0.]), \
                                                  translation = np.array([0., 0., 0.1065]) ))

pRe = gc.Point(Rec6.transformation.transx, Rec6.transformation.transy, Rec6.transformation.transz)

# Generation of heliostats with facets
H0 = Heliostat(n_facets_x = 2, n_facets_y = 2, helio_size_x = 0.00945, helio_size_y = 0.01284)
LOBJ6 = generate_h_a(theta_deg=SZA, phi_deg= PHI, receiver_pos=pRe, min_ang_deg=140, max_ang_deg=220, gap_ang_deg = 10, first_dist=0.15,
                        n_heliostats = 4, gap_dist = 0.03, pillar_height= 0.00517, reflectivity=0.88, roughness = 0.001, heliostat_type=H0)

# Without forgetting to add the reveicer at the list of objects
LOBJ6.append(Rec6)

### Drawing verification

In [ ]:
# visualize without facets
LOBJ6_bis = generate_h_a(theta_deg=SZA, phi_deg= PHI, receiver_pos=pRe, min_ang_deg=140, max_ang_deg=220, gap_ang_deg = 10, first_dist=0.15,
                        n_heliostats = 4, gap_dist = 0.03, pillar_height= 0.00517, helio_size_x = 0.00945, helio_size_y = 0.01284, reflectivity=0.88, roughness = 0.001)
LOBJ6_bis.append(Rec6)
fig1 = visualize_entity(LOBJ6_bis, theta_deg=SZA, phi_deg=PHI)

### Run the simulation

In [ ]:
NormalRec = gc.Vector(0, 0, 1)
TT = gc.get_rotate_y_tf(Rec6.transformation.rotation[1])
NormalRec = TT(NormalRec)
NormalRec = gc.normalize(NormalRec)

Pmin = [-0.6, -0.6, 0.]
Pmax = [0.6, 0.6, 0.03]
interval0 = [Pmin, Pmax] # enable to earn some computational time, can be very usefull in FF mode
aer = AerOPAC('desert', 0.5, 550.)
pro = Atm1D('afglms', comp=[aer])
v_sun = gc.ang2vec(SZA, PHI, vec_view='nadir')
# sun_fov gives the sun its angular size here, where the run has no
# local estimate; with the le parameter it is le_fov=0.266 instead
lMode = CusBackward(pos=pRe, v=NormalRec, aldeg=90, lmode="BR", rec=Rec6,
                    v_sun=v_sun, sun_fov=0.266)
m6 = Smartg(double = True, obj3d=True, back=True).run( surf = LambSurface(alb=AlbedoCst(0.25)), 
        n_f=1e6, wavelength=550., nb_photons=1e9, nb_loop=1e7, atm=pro,
        my_objects=LOBJ6, interval = interval0, cus_l=lMode, direct=True)
m6_ds = m6


### Results

In [ ]:
print("kernel time(s)=", m6.attrs['kernel time (s)'])
cat_view(m6_ds, output_unit='FLUX', flux_unit='kW')

In [ ]:
receiver_view(ds_sg_out=m6_ds, vmin=0, flux_unit='kW')

# 2D Coastal Bathymetry — depth-varying ocean reflectance

This section demonstrates how ocean **depth** drives spectral reflectance changes along a coastal transect.

## Physical concept

In a shallow-water scenario:
- Photons enter the ocean at the surface, travel through the water column, reflect off the seafloor, and exit.
- The water column absorbs and scatters photons as a function of depth and wavelength.
- **Shallow water** → stronger seafloor contribution → higher reflectance, less spectrally selective.
- **Deep water** → seafloor signal attenuated exponentially → lower reflectance, selective (blue > red, because water absorbs red faster).

## Scene setup

```
x < 0 km : Land   (Lambertian, albedo = 0.10)
x > 0 km : Ocean  (FlatSurface + HydrosolPR, sandy seafloor albedo = 0.20)
```

The ocean depth increases linearly from the coast (`d_coast` at x=0) to `d_max` at x=5 km.
Because SmartG's ocean is 1D (single water profile per run), three separate runs are performed at
representative depths **d = 5 m**, **20 m**, and **100 m** and composited into a synthetic transect.

In [ ]:
# Additional imports needed for this section
import matplotlib.pyplot as plt
from smartg.surface import RoughSurface, Environment
from smartg.albedo import AlbedoMap
from smartg.water import Water1D, HydrosolPR

In [ ]:
# ── Scene parameters ──────────────────────────────────────────────────────────
D_km   = 5.          # half-domain extent [km]
N      = 11          # number of sensors in the transect
depths = [5., 20., 100.]   # ocean depths to simulate [m]

# Wavelengths: three key ocean-colour bands
wavelength_bath = np.array([490., 550., 670.])   # [nm]

# ── Coastal environment (AlbedoMap, env=5) ───────────────────────────────────
#   x < 0 : land  (environment index 0 → AlbedoCst(0.10))
#   x ≥ 0 : ocean (environment index -1 → uses surf + water from run())
#
# AlbedoMap x-bins: (-inf, 0] = land, (0, +inf) = ocean
x_env = np.array([0., 1e8])   # [km] upper edges of x-bins
y_env = np.array([1e8])        # single y-bin (whole domain)
Ai_coast = np.array([[0],      # bin x<=0  : land (Alb_list index 0)
                     [-1]])    # bin x>0   : ocean (negative → surf+water)
Alb_land  = AlbedoCst(0.10)
ALB_coast = AlbedoMap(Ai_coast, x_env, y_env, [Alb_land])
env_coast = Environment(env=5, x0=0., y0=0., alb=ALB_coast)

# ── Surface and atmosphere ────────────────────────────────────────────────────
surf_ocean = RoughSurface()                        # flat air/water interface
atm_bath   = Atm1D('afglt')                     # standard mid-latitude atmosphere

# ── Sensor transect: nadir-looking sensors from x=-4 km to x=+4 km ───────────
x_sensors = np.linspace(-D_km + 0.5, D_km - 0.5, N)   # [km]
sensors_bath = [Sensor(POSZ=120., POSX=xi, POSY=0., LOC='ATMOS', THDEG=180.)
                for xi in x_sensors]

# Sun zenith 30°, azimuth 0°
le_bath = {'th_deg': np.array([30.]), 'phi_deg': np.array([0.]), 'zip': True}

print(f"Transect x positions [km]: {x_sensors}")
print(f"Ocean depths to simulate [m]: {depths}")

In [ ]:
# ── Run one simulation per depth ─────────────────────────────────────────────
#
#  ocean model:  HydrosolPR(chl=0.1)  — coastal turbid water (chl = 0.1 mg/m³)
#  seafloor:     AlbedoCst(0.20) — sandy/coral bottom (20% reflectance)
#  alb in Water1D is the *seafloor* albedo; grid sets the water-column thickness.
#
#  Note: SmartG's ocean is 1-D (single global water profile per run), so we run once per
#  representative depth and then assemble a synthetic spatial transect.

res_bath = {}   # key = depth [m]

for d in depths:
    water_d = Water1D(grid=[0., -d], comp=[HydrosolPR(chl=0.1)], alb=AlbedoCst(0.20))
    m_d = Smartg().run(
        wavelength         = wavelength_bath,
        le         = le_bath,
        sensor     = sensors_bath,
        atm        = atm_bath,
        surf       = surf_ocean,
        env        = env_coast,
        water      = water_d,
        nb_photons=5e7,
        beer=1,
        r_r=0,
    )
    m_d = drop_axes(m_d, 'Zenith angles')
    res_bath[d] = m_d
    print(f"depth = {d:6.1f} m  →  mean ocean reflectance @ 490 nm: "
          f"{m_d['I_up (TOA)'].data[x_sensors > 0, 0].mean():.4f}")

### Visualise the depth-dependent reflectance

Expected physics:
- **Left plot**: land pixels (x < 0) are spectrally flat at ~0.10; ocean pixels decrease and become more blue-dominant as depth increases.
- **Right plot / depth curve**: at 670 nm (red, strong water absorption) the bottom signal vanishes quickly; at 490 nm (blue, low water absorption) the seafloor remains visible at greater depths.
- The coastline transition (x = 0) shows an abrupt change between land and ocean surface types.

In [ ]:
# ── Visualisation ─────────────────────────────────────────────────────────────
#
# Left panel  : reflectance transect (land → ocean) at three wavelengths, one
#               line per depth.
# Right panel : reflectance spectra at x=+2 km (ocean pixel) for each depth.

colors_d = {5.: '#1f77b4', 20.: '#ff7f0e', 100.: '#2ca02c'}
ls_wl    = {490.: '-',  550.: '--', 670.: ':'}
labels_wl = {490.: '490 nm', 550.: '550 nm', 670.: '670 nm'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── LEFT : spatial transect ───────────────────────────────────────────────────
ax = axes[0]
for d in depths:
    m = res_bath[d]
    R = m['I_up (TOA)'].data           # shape (N_sensors, N_wl)
    for i_w, wavelength_nm in enumerate(wavelength_bath):
        lbl = f"{int(wavelength_nm)} nm, d={int(d)} m" if wavelength_nm == 490. else None
        ax.plot(x_sensors, R[:, i_w],
                color=colors_d[d],
                ls=ls_wl[wavelength_nm],
                label=lbl)

ax.axvline(0, color='k', lw=0.8, ls='--', label='coastline')
ax.axvspan(-D_km, 0,  alpha=0.08, color='saddlebrown', label='land')
ax.axvspan(0, D_km,   alpha=0.08, color='steelblue',   label='ocean')
ax.set_xlabel('Cross-shore position [km]')
ax.set_ylabel('Reflectance  I/F  [–]')
ax.set_title('Reflectance transect (SZA=30°, nadir sensor)')
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)

# ── RIGHT : depth vs reflectance spectra (ocean only) ────────────────────────
ax2 = axes[1]
# Pick the sensor closest to x=+2 km (ocean pixel)
i_oc = np.argmin(np.abs(x_sensors - 2.0))
for d in depths:
    m = res_bath[d]
    R_oc = m['I_up (TOA)'].data[i_oc, :]   # shape (N_wl,)
    ax2.plot(wavelength_bath, R_oc, 'o-', color=colors_d[d], label=f"depth = {int(d)} m")

ax2.set_xlabel('Wavelength [nm]')
ax2.set_ylabel('Reflectance  I/F  [–]')
ax2.set_title(f'Spectral signature at x≈{x_sensors[i_oc]:.1f} km (ocean pixel)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ── Depth–reflectance curves at fixed wavelengths ─────────────────────────────
fig2, ax3 = plt.subplots(figsize=(6, 4))
i_oc = np.argmin(np.abs(x_sensors - 2.0))
for i_w, wavelength_nm in enumerate(wavelength_bath):
    R_vs_d = [res_bath[d]['I_up (TOA)'].data[i_oc, i_w] for d in depths]
    ax3.plot(depths, R_vs_d, 'o-', label=labels_wl[wavelength_nm])

ax3.set_xscale('log')
ax3.set_xlabel('Ocean depth [m]')
ax3.set_ylabel('Reflectance  I/F  [–]')
ax3.set_title('Depth–reflectance relationship (coastal turbid water, seafloor alb = 0.20)')
ax3.legend()
ax3.grid(alpha=0.3)
plt.tight_layout()
plt.show()